# QSEncode-Insight: Resource-Aware State Preparation

**Audience:** contest reviewers and PyQPanda users. **Prerequisites:** a source checkout with PyQPanda3, NumPy, and SciPy.

**Goals:** compare a refusal with a compression recommendation, inspect candidates and resources, prepare a runnable program, and distinguish standard from audit verification.

## Outline

1. Load the sealed Gaussian N=8 example.
2. Compare Walsh refusal with Fourier compression.
3. Inspect candidates, resources, and the prepared artifact.
4. Run a five-repeat audit and check EvidenceScope.

In [ ]:
from pathlib import Path
import sys
import numpy as np

# Support Run All from the repository root or this example directory.
for root in (Path.cwd(), *Path.cwd().parents):
    source_root = root / 'pyqpanda-algorithm'
    if (source_root / 'pyqpanda_alg').is_dir():
        sys.path.insert(0, str(source_root)); break
    if (root / 'pyqpanda_alg').is_dir():
        sys.path.insert(0, str(root)); break

from pyqpanda_alg.QSEncode import QSEncodeInsight

probabilities = np.array([
    0.0006917643261373052, 0.015724004731018214,
    0.1261730210273901, 0.3574112099154543,
    0.3574112099154544, 0.1261730210273902,
    0.01572400473101823, 0.0006917643261373052,
])
float(probabilities.sum()), len(probabilities)

## 1. Walsh mode refuses compression

A smaller transformed representation is not automatically a cheaper compiled program.

In [ ]:
walsh_engine = QSEncodeInsight(basis='walsh')
walsh_result = walsh_engine.analyze(probabilities)
{'decision': walsh_result.selection.decision.value,
 'k_star': walsh_result.error_budget.k_star,
 'baseline_2q': walsh_result.selection.baseline_resource.compiled_two_qubit_gates,
 'baseline_depth': walsh_result.selection.baseline_resource.compiled_depth}

## 2. Fourier mode selects sparse preparation

The basis is explicit; v1 does not compare Walsh and Fourier automatically.

In [ ]:
fourier_engine = QSEncodeInsight(basis='fourier')
fourier_result = fourier_engine.analyze(probabilities)
{'decision': fourier_result.selection.decision.value,
 'k_star': fourier_result.error_budget.k_star,
 'winner': fourier_result.selection.selected_candidate_id,
 'method': fourier_result.selection.method.value}

## 3. Candidate table and resource attribution

Incompatible candidates remain visible. Only k=4 is displayed to keep output small.

In [ ]:
candidate_table = []
for candidate in fourier_result.candidates:
    if candidate.k == 4:
        audit = candidate.resource_audit
        candidate_table.append({
            'candidate': candidate.candidate_id,
            'compatible': candidate.capability.compatible,
            'eligible': candidate.eligible,
            '2q': None if audit is None else audit.compiled_two_qubit_gates,
            'depth': None if audit is None else audit.compiled_depth,
            'reason': candidate.eligibility_reason})
candidate_table

In [ ]:
attr = fourier_result.attribution
{'2q_total_truncation_preparation': (attr.total_two_qubit_difference, attr.truncation_two_qubit_difference, attr.preparation_two_qubit_difference),
 'depth_total_truncation_preparation': (attr.total_depth_difference, attr.truncation_depth_difference, attr.preparation_depth_difference)}

## 4. Prepare a runnable program

InsightResult remains JSON-safe; the QProg lives in a separate artifact.

In [ ]:
artifact = fourier_engine.prepare(probabilities, result=fourier_result)
{'candidate': artifact.selected_candidate_id, 'k': artifact.k,
 'output_qubits': artifact.output_qubits, 'ancillas': artifact.ancillas,
 'verification': artifact.verification_status}

## 5. Audit verification and EvidenceScope

Standard does not claim compiled semantic certification. Audit reuses the five compiled attempts and requires 5/5 passes.

In [ ]:
audit_result = QSEncodeInsight(basis='fourier', verification='audit').analyze(probabilities)
{'status': audit_result.semantic_verification.status,
 'attempts': len(audit_result.semantic_verification.attempts),
 'minimum_fidelity': audit_result.semantic_verification.minimum_fidelity,
 'evidence_scope': audit_result.evidence_scope.status.value}

## Exercise

Predict the EvidenceScope status for `fidelity_target=0.98`. The answer scaffold below avoids extra compilation during the default Run All; uncomment the analysis when you want to verify it.

In [ ]:
# outside = QSEncodeInsight(basis='walsh', fidelity_target=0.98).analyze(probabilities)
# outside.evidence_scope.status.value, outside.evidence_scope.reasons
expected_status = 'outside_validated_scope'
expected_status

## Limitations

- These are compiled-resource results, not hardware speedup evidence.
- The preregistered validation scope is N<=64; Dirichlet inputs were weaker.
- v1 has no automatic cross-basis selector.
- Large OriginIR, statevectors, and Locked benchmark artifacts are intentionally not embedded.